# Bài 3 — Sinh câu trả lời có trích dẫn

**Tuần 21** · Kiến thức dùng ở `ex03_eval_rag.py` và **P6**

Hai bài trước lo phần **tìm**. Bài này lo phần **trả lời** — và phần khó nhất
của nó không phải viết prompt, mà là:

> **Làm sao biết câu trả lời thật sự đến từ tài liệu, chứ không phải model bịa?**

> 💡 Notebook chạy được **không cần API key**: các ô gọi API sẽ tự bỏ qua và in
> ra kết quả minh hoạ để bạn vẫn theo được mạch bài.

In [ ]:
import sys
from pathlib import Path

import numpy as np

# Tim thu muc goc du an du notebook duoc mo tu dau
GOC = Path.cwd()
while not (GOC / "pyproject.toml").exists() and GOC != GOC.parent:
    GOC = GOC.parent
P08 = GOC / "curriculum" / "08-rag"
sys.path.insert(0, str(P08))

from tien_ich import bat_utf8, nap_tai_lieu, nap_cau_hoi_vang
from nhung_gia_lap import nhung, nhung_nhieu, giai_thich, SO_CHIEU, TEN_CHIEU

bat_utf8()   # Windows: console cp1252 se lam sap moi lenh print tieng Viet

DOCS = nap_tai_lieu()
VANG = nap_cau_hoi_vang()

print(f"{len(DOCS)} tài liệu · {sum(len(v) for v in DOCS.values()):,} ký tự")
print(f"{len(VANG)} câu hỏi vàng · embedding giả lập {SO_CHIEU} chiều")

In [ ]:
def cat_co_dinh(van_ban, kich_thuoc, chong_lap=0):
    """Giong ex01_chunking.cat_co_dinh - dat o day de notebook chay doc lap."""
    if chong_lap >= kich_thuoc:
        raise ValueError("chong_lap phai nho hon kich_thuoc")
    buoc = kich_thuoc - chong_lap
    return [van_ban[i:i + kich_thuoc] for i in range(0, len(van_ban), buoc)]


def cat_kho(docs, kich_thuoc=600, chong_lap=120):
    """Cat toan bo kho -> (danh sach chunk, danh sach ten file tuong ung)."""
    chunks, nguon = [], []
    for ten, noi_dung in docs.items():
        for c in cat_co_dinh(noi_dung, kich_thuoc, chong_lap):
            chunks.append(c)
            nguon.append(ten)
    return chunks, nguon


def gop(s):
    """Gop moi khoang trang/xuong dong thanh mot dau cach."""
    return " ".join(s.split())


def co_trich_dan(cac_chunk, trich_dan):
    """Doan trich dan co nam trong cac chunk lay ve khong?"""
    kho = gop(" || ".join(cac_chunk))
    return gop(trich_dan) in kho

In [ ]:
import os

import anthropic

MODEL_RE = "claude-haiku-4-5"          # KHÔNG thêm hậu tố ngày tháng

CO_KEY = bool(os.environ.get("ANTHROPIC_API_KEY"))
client = anthropic.Anthropic() if CO_KEY else None


def goi(**kwargs):
    """Gọi API nếu có key; chưa có thì in thông báo và trả về None."""
    if not CO_KEY:
        print("⏭  Bỏ qua — chưa có ANTHROPIC_API_KEY (xem SETUP.md mục 5)")
        return None
    return client.messages.create(**kwargs)


def van_ban(response) -> str:
    """Lấy toàn bộ text. KHÔNG dùng response.content[0].text!"""
    if response is None:
        return ""
    return "".join(b.text for b in response.content if b.type == "text")


print("Có API key" if CO_KEY else "Chưa có API key — notebook vẫn chạy hết")

In [ ]:
# Dựng lại phần tìm kiếm của bài 2, gọn hơn
import math
import re
from collections import Counter

MAU_TU = re.compile(r"[0-9a-zà-ỹđ]+(?:[-.][0-9a-zà-ỹđ]+)*")
tach_tu = lambda s: MAU_TU.findall(s.lower())

CHUNKS, NGUON = cat_kho(DOCS, 600, 120)
M = nhung_nhieu(CHUNKS)

_tu = [tach_tu(c) for c in CHUNKS]
_tf = [Counter(t) for t in _tu]
_len = [len(t) for t in _tu]
_ltb = sum(_len) / len(_len)
_df = Counter()
for t in _tu:
    _df.update(set(t))


def diem_bm25(q, k1=1.5, b=0.75):
    qt = tach_tu(q)
    out = []
    for i in range(len(CHUNKS)):
        chuan = 1 - b + b * _len[i] / _ltb
        s = 0.0
        for w in qt:
            tf = _tf[i].get(w, 0)
            if tf:
                idf = math.log(1 + (len(CHUNKS) - _df[w] + 0.5) / (_df[w] + 0.5))
                s += idf * tf * (k1 + 1) / (tf + k1 * chuan)
        out.append(s)
    return np.array(out)


def tim(cau, k=3, k0=60):
    """Hybrid RRF trên toàn bộ hai bảng xếp hạng."""
    a = np.argsort(-diem_bm25(cau))
    b = np.argsort(-(M @ nhung(cau)))
    diem = {}
    for bang in (a, b):
        for hang, i in enumerate(bang, 1):
            diem[int(i)] = diem.get(int(i), 0.0) + 1.0 / (k0 + hang)
    top = sorted(diem, key=lambda i: (-diem[i], i))[:k]
    return [CHUNKS[i] for i in top], [NGUON[i] for i in top]


doan, nguon = tim("Nhân viên có bao nhiêu ngày phép năm?")
print(f"Lấy {len(doan)} đoạn từ: {nguon}")

## 1. Ngữ cảnh phải được ĐÁNH SỐ

Không đánh số thì model **không có cách nào** trỏ tới một đoạn cụ thể, và bạn
không có cách nào kiểm tra câu trả lời đến từ đâu.

Đánh số không phải để cho đẹp — nó là **điều kiện cần** để chống bịa đặt.

In [ ]:
def tao_ngu_canh(cac_doan, cac_nguon):
    return "\n\n".join(
        f"[{i}] ({ng})\n{doan}"
        for i, (doan, ng) in enumerate(zip(cac_doan, cac_nguon), start=1)
    )


ngu_canh = tao_ngu_canh(doan, nguon)
print(ngu_canh[:600], "...")

## 2. Prompt cho RAG

Ba mệnh lệnh, thiếu cái nào cũng hỏng:

In [ ]:
HUONG_DAN = """Bạn trả lời câu hỏi của nhân viên dựa trên tài liệu nội bộ.

QUY TẮC BẮT BUỘC:
1. CHỈ dùng thông tin trong các đoạn được cung cấp. Tuyệt đối không dùng
   kiến thức bên ngoài, kể cả khi bạn chắc chắn là đúng.
2. Mỗi ý phải kèm số đoạn dạng [n]. Chỉ dùng các số đoạn CÓ THẬT.
3. Nếu các đoạn không chứa đủ thông tin, trả lời đúng câu:
   "Không tìm thấy thông tin trong tài liệu."
   rồi nói ngắn gọn tài liệu có gì gần với câu hỏi.

Trả lời ngắn gọn, bằng tiếng Việt."""


def tra_loi(cau_hoi, k=3):
    cac_doan, cac_nguon = tim(cau_hoi, k)
    nc = tao_ngu_canh(cac_doan, cac_nguon)
    r = goi(
        model=MODEL_RE,
        max_tokens=600,
        system=HUONG_DAN,
        messages=[{"role": "user",
                   "content": f"<tai_lieu>\n{nc}\n</tai_lieu>\n\nCâu hỏi: {cau_hoi}"}],
    )
    return van_ban(r), cac_doan, cac_nguon


cau = "Nhân viên chính thức có bao nhiêu ngày phép năm?"
tl, cac_doan, cac_nguon = tra_loi(cau)
print(tl or "   (minh hoạ) Nhân viên chính thức có 12 ngày phép năm [1].")

Ba quy tắc, mỗi cái chặn một kiểu hỏng:

| Quy tắc | Chặn gì |
|---|---|
| Chỉ dùng đoạn được cung cấp | Model trả lời bằng kiến thức nền — nghe đúng nhưng không phải chính sách công ty bạn |
| Trích dẫn `[n]` | Không kiểm chứng được nguồn |
| Câu từ chối cố định | Model bịa khi không tìm thấy |

> 📌 Quy tắc 3 dùng **một câu từ chối cố định**, không phải "hãy nói bạn không
> biết". Cố định thì code của bạn **so khớp được** — đó là cách bài tập 3 phát
> hiện hệ thống có từ chối đúng lúc hay không.

## 3. Kiểm tra trích dẫn bằng CODE, không bằng cảm giác

In [ ]:
MAU_TRICH_DAN = re.compile(r"\[([1-9][0-9]*)\]")


def kiem_tra_trich_dan(cau_tra_loi, so_doan):
    cac_so = {int(s) for s in MAU_TRICH_DAN.findall(cau_tra_loi)}
    ngoai = {n for n in cac_so if n > so_doan}
    return {"cac_so": cac_so, "ngoai_pham_vi": ngoai,
            "hop_le": bool(cac_so) and not ngoai}


VI_DU = {
    "tốt":            "Phép năm là 12 ngày [1]. Chuyển sang năm sau tối đa 5 ngày [2].",
    "không dẫn nguồn": "Phép năm là 12 ngày mỗi năm.",
    "BỊA trích dẫn":   "Phép năm là 12 ngày [1], và được cộng dồn vô hạn [7].",
}

for ten, tl_vd in VI_DU.items():
    kq = kiem_tra_trich_dan(tl_vd, so_doan=3)
    print(f'{ten:<18} hợp lệ={str(kq["hop_le"]):<6} '
          f'trích dẫn={sorted(kq["cac_so"])}  ngoài phạm vi={sorted(kq["ngoai_pham_vi"])}')

> ⚠️ **Dòng thứ ba là kiểu bịa đặt nguy hiểm nhất.** Model viết `[7]` khi bạn
> chỉ đưa 3 đoạn. Câu trả lời trông **càng đáng tin** vì có số trích dẫn — và
> đúng vì thế mà người đọc sẽ không đi kiểm tra.
>
> Một dòng regex bắt được lỗi này. Không có nó, bạn phải tin vào cảm giác.

## 4. Câu hỏi KHÔNG có đáp án — phép thử thật sự

Đây là chỗ phân biệt hệ thống nghiêm túc với hệ thống biết bịa.

Câu *"công ty có xe đưa đón không?"* **không có** trong tài liệu. Nhưng kho lại
có "phụ cấp đi lại" — rất giống. Retrieval **chắc chắn** trả về đoạn đó với
điểm cao.

In [ ]:
cau_bay = "Công ty có xe đưa đón nhân viên đi làm không?"
doan_bay, nguon_bay = tim(cau_bay, 3)

print("Retrieval trả về (đều KHÔNG chứa đáp án):")
for i, (d, ng) in enumerate(zip(doan_bay, nguon_bay), 1):
    print(f"   [{i}] ({ng}) {gop(d)[:75]}...")

print()
tl_bay, _, _ = tra_loi(cau_bay)
if tl_bay:
    print(tl_bay)
else:
    print("   (minh hoạ) Không tìm thấy thông tin trong tài liệu.")
    print("   Các đoạn trên nói về phụ cấp và quy trình nghỉ phép,")
    print("   không đoạn nào nói về xe đưa đón.")

**Retrieval không có cách nào "đúng" ở đây** — nó buộc phải trả về đoạn gì đó.
Trách nhiệm nằm hoàn toàn ở bước sinh câu trả lời.

> 📌 Đây là lý do bộ câu hỏi vàng **bắt buộc** phải có loại `khong_co`. Một hệ
> thống trả lời được mọi câu là một hệ thống biết bịa — và nếu bộ test của bạn
> chỉ có câu trả lời được, bạn sẽ không bao giờ biết điều đó.

## 5. Citations có sẵn của Claude API

Cách đánh số ở trên là thủ công. Claude hỗ trợ trích dẫn **ở mức API**: bật
`citations` trên khối `document`, model trả về từng khối text kèm **vị trí ký tự
chính xác** trong tài liệu gốc.

Ưu điểm: vị trí do **API** trả về, model không tự bịa số được.

In [ ]:
def tra_loi_citations(cau_hoi, k=3):
    cac_doan, cac_nguon = tim(cau_hoi, k)
    noi_dung = [
        {"type": "document",
         "source": {"type": "text", "media_type": "text/plain", "data": d},
         "title": ng,
         "citations": {"enabled": True}}          # bật cho MỘT thì phải bật cho TẤT CẢ
        for d, ng in zip(cac_doan, cac_nguon)
    ]
    noi_dung.append({"type": "text", "text": cau_hoi})

    return goi(model=MODEL_RE, max_tokens=600,
               system="Trả lời ngắn gọn bằng tiếng Việt, dựa hoàn toàn vào tài liệu.",
               messages=[{"role": "user", "content": noi_dung}])


r = tra_loi_citations("Nhân viên chính thức có bao nhiêu ngày phép năm?")

if r is None:
    print("   (minh hoạ) Mỗi khối text trả về kèm mảng citations:")
    print('   text: "Nhân viên chính thức có 12 ngày phép năm."')
    print('   citations[0]: document_title="chinh_sach_nghi_phep.md"')
    print('                 cited_text="Nhân viên chính thức có 12 ngày phép năm."')
    print("                 start_char_index=41  end_char_index=88")
else:
    for khoi in r.content:
        if khoi.type != "text":
            continue
        print(khoi.text)
        for tr in (getattr(khoi, "citations", None) or []):
            print(f"      ↳ {tr.document_title}: {tr.cited_text[:60]!r}")

> ⚠️ **Hai ràng buộc phải nhớ:**
>
> 1. Bật `citations` cho **một** document thì phải bật cho **tất cả** — nếu không API báo lỗi.
> 2. Citations **không dùng chung được với structured output** (`output_config.format`) — API trả **400**. Cần cả hai thì phải tách làm hai lượt gọi.
>
> Ràng buộc thứ hai hay làm hỏng kế hoạch "trả về JSON có trường `trich_dan`".
> Biết trước sẽ đỡ mất một buổi chiều.

## 6. Nhiều đoạn hơn = tốt hơn? Không hẳn

Bài 1 đã cho thấy recall tăng theo k. Nhưng **chất lượng câu trả lời không đi
theo recall**, vì hai lý do:

- **Lost in the middle** — model chú ý phần **đầu** và **cuối** ngữ cảnh nhiều
  hơn phần **giữa**. Đoạn đúng nằm giữa 10 đoạn dễ bị bỏ qua.
- **Context stuffing** — đoạn rác làm loãng đoạn đúng.

Hãy nhìn cái giá trước đã.

In [ ]:
GIA_VAO = 1.00 / 1_000_000

print(f'{"k":>3}{"ký tự ngữ cảnh":>17}{"~token":>9}{"$/1.000 câu":>14}')
for k in (1, 3, 5, 10, 20):
    d, ng = tim("Nhân viên có bao nhiêu ngày phép năm?", k)
    ky_tu = len(tao_ngu_canh(d, ng))
    tok = ky_tu / 3
    print(f"{k:>3}{ky_tu:>17,}{tok:>9,.0f}{tok * GIA_VAO * 1000:>14.2f}")

In [ ]:
# Khi k=10, đoạn chứa đáp án thường nằm ở vị trí nào trong ngữ cảnh?
vi_tri = Counter()
for c in VANG:
    if not c["trich_dan"]:
        continue
    d10, _ = tim(c["cau_hoi"], 10)
    for i, doan_i in enumerate(d10, 1):
        if any(gop(td) in gop(doan_i) for td in c["trich_dan"]):
            vi_tri[i] += 1
            break
    else:
        vi_tri[0] += 1                      # không lấy được

print("Vị trí của đoạn đúng ĐẦU TIÊN trong top-10:")
for v in sorted(vi_tri):
    nhan = "không lấy được" if v == 0 else f"vị trí {v}"
    canh_bao = "   <- vùng GIỮA, dễ bị bỏ qua" if 3 < v < 8 else ""
    print(f"   {nhan:<16} {chr(9608) * vi_tri[v]} {vi_tri[v]}{canh_bao}")

giua = sum(n for v, n in vi_tri.items() if 3 < v < 8)
tong = sum(vi_tri.values())
print()
print(f"{giua}/{tong} câu có đáp án rơi vào vùng giữa ngữ cảnh khi k=10.")

Trên kho nhỏ này, phần lớn câu hỏi có đáp án nằm ngay vị trí 1 — nên
**lost in the middle** hầu như chưa gây hại. Đó là một kết quả trung thực, không
phải một lời quảng cáo cho việc tăng k.

Điều đáng lo nằm ở chỗ khác: nhìn lại bảng chi phí phía trên. Từ k=3 lên k=10,
bạn trả **gấp 3,3 lần** cho mỗi câu hỏi, trong khi bảng vị trí cho thấy 13/16 câu
đã có đáp án ở ngay vị trí 1. **Bạn đang mua thêm rất ít với rất nhiều tiền.**

> 📌 **Đây là chỗ dễ đi sai hướng nhất trong RAG.** Bạn tăng k, thấy recall tăng,
> kết luận hệ thống tốt lên — trong khi chi phí tăng gấp ba và chất lượng câu trả
> lời có thể **tệ đi** vì đoạn đúng bị chôn giữa đống đoạn rác.
>
> Với kho lớn hơn và câu hỏi mơ hồ hơn, tỷ lệ rơi vào vùng giữa sẽ cao lên nhiều.
> Nếu bạn chỉ đo recall, bạn sẽ đi sai hướng một cách rất tự tin. Bài 4 đo cả hai
> đầu để chuyện đó không xảy ra.

**Cách xử lý:** lấy rộng (k=10–20) → **rerank** → chỉ đưa 3–5 đoạn tốt nhất vào
prompt, và xếp đoạn liên quan nhất **lên đầu**.

## Tóm tắt

| Điều | Nhớ gì |
|---|---|
| **Đánh số đoạn** | Điều kiện cần để kiểm chứng, không phải để trang trí |
| **Câu từ chối cố định** | Cố định thì code so khớp được; "tuỳ ý" thì không |
| **Trích dẫn ngoài phạm vi** | Dạng bịa nguy hiểm nhất — bắt bằng regex, không bằng cảm giác |
| **Câu không có đáp án** | Retrieval buộc phải trả về gì đó; trách nhiệm ở bước sinh |
| **Citations API** | Vị trí do API trả về; nhưng đụng độ với structured output |
| **k lớn hơn** | Recall tăng, chất lượng có thể giảm. Đo cả hai |

**Bài tập:** `curriculum/08-rag/exercises/ex03_eval_rag.py` (mục 3.1–3.3)

➡️ Bài tiếp: **[04 — Đo chất lượng RAG](04_danh_gia_rag.ipynb)** ⭐⭐